In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, explode_outer, from_unixtime
from delta.tables import DeltaTable

########################################
########---Sales Orders-----########
########################################

def sales_orders(df):
    sales_orders = df.select(
        "order_number",
        "customer_id",
        "customer_name",
        "number_of_line_items",
        from_unixtime(col("order_datetime")).alias("order_timestamp")
    ).distinct()
    return sales_orders


########################################
########---Ordered Products-----########
########################################
def ordered_products(df):

    ordered_product_schema = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("promotion_info", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
    ]))

    df_ordered_products = df.withColumn("ordered_products", from_json(col("ordered_products"), ordered_product_schema ))\
                    .withColumn("ordered_products", explode_outer("ordered_products"))\
                    .select(
                        "customer_id",
                        "customer_name",
                        "order_number",
                        "ordered_products.id",
                        col("ordered_products.name").alias("product_name"),
                        "ordered_products.price",
                        "ordered_products.curr",
                        "ordered_products.qty",
                        "ordered_products.unit"
                        ).distinct()
    
    return df_ordered_products

########################################
########---promotions-----##############
########################################
def promotions(df):
    promo_schema = ArrayType(
        StructType([
            StructField("promo_disc", DoubleType(), True),
            StructField("promo_id", StringType(), True),
            StructField("promo_item", StringType(), True),
            StructField("promo_qty", StringType(), True)
        ])
    )

    df_promo = df.withColumn( "promo_info",from_json(col("promo_info"), promo_schema)) \
                .withColumn("promo_info", explode_outer("promo_info"))\
                .filter(col("promo_info").isNotNull())\
                .select(
                        "customer_id",
                        "order_number",
                        col("promo_info.promo_id").alias("promo_id"),
                        col("promo_info.promo_item").alias("promo_product_id"),
                        col("promo_info.promo_disc").alias("discount"),
                        col("promo_info.promo_qty").cast("int").alias("promo_quantity"))\
                .dropDuplicates()
                
    return df_promo

########################################
########---promotions-----##############
########################################

def clicked_items(df):
    clicked_item_schema = ArrayType(
        ArrayType(StringType())
    )

    clicked_items = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_item_schema))\
                    .withColumn("clicked_items", explode_outer("clicked_items"))\
                    .select(
                        "customer_id",
                        "customer_name",
                        "order_number",
                        col("clicked_items")[0].alias("product_id"),
                        col("clicked_items")[1].alias("score")
                    ).distinct()

    return clicked_items


def scd_merge_table(spark, source_table, target_table, business_key):

    if not spark.catalog.tableExists(target_table):
        print("First Load: Creating Silver Table", target_table)
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print("Table Created")

    else:
        print("Incremental Load: Performing SCD Type 1 Merge",target_table)

        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )


        delta_table.alias("target").merge(source_table.alias("source"), 
                                        merge_condition
                                        ).whenMatchedUpdateAll()\
                                            .whenNotMatchedInsertAll()\
                                            .execute()
        print("Merge Successfully Completed")

##############################
#####--Main Code----######
##############################

print(f"Reading source table")
bronze_table = "ecommerce_analytics.bronze.sales_orders"
df = spark.read.table(bronze_table)

#Transformations
print(f"Transforming data Started")
sales_orders_df = sales_orders(df)
ordered_products_df = ordered_products(df)
promotions_df = promotions(df)
clicked_items_df = clicked_items(df)


#Write
scd_merge_table(spark, sales_orders_df,  "ecommerce_analytics.silver.sales_orders",["order_number", "number_of_line_items"])
scd_merge_table(spark, ordered_products_df, "ecommerce_analytics.silver.order_products",["order_number","id","price"])
scd_merge_table(spark, promotions_df, "ecommerce_analytics.silver.promotions",["order_number", "promo_id", "promo_product_id", "promo_quantity"])
scd_merge_table(spark, clicked_items_df, "ecommerce_analytics.silver.clicked_items",["order_number","product_id","score"])

print(f"Successfully wrote all tables")